In [1]:
import os
os.environ['KERAS_BACKEND'] = "torch"

import numpy as np
import keras
from keras.datasets import imdb
from keras.utils import pad_sequences
from keras.models import load_model

In [2]:
model = load_model('simple_rnn_imdb.keras')

In [3]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (5096, 300, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (5096, 128)            │        22,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (5096, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569,125 (5.99 MB)

 Trainable params: 523,041 (2.00 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,046,084 (3.99 MB)

In [4]:
model.get_weights()

[array([[-0.01378886, -0.01387029, -0.01727378, ..., -0.03859258,
         -0.04783667,  0.01309641],
        [ 0.00196639,  0.03695622, -0.03312319, ..., -0.04353299,
         -0.03495622,  0.03449311],
        [-0.01482518, -0.03667568, -0.02051487, ...,  0.0275545 ,
         -0.02539544,  0.00762434],
        ...,
        [ 0.01187681,  0.05868157, -0.06423093, ...,  0.04766762,
         -0.01309617, -0.00038558],
        [ 0.02608489, -0.05043023,  0.03636299, ...,  0.01283969,
         -0.02142925, -0.01580557],
        [-0.00528778,  0.05617181, -0.03190489, ...,  0.04984381,
          0.05724933, -0.00714667]], shape=(10000, 50), dtype=float32),
 array([[-0.0910047 , -0.14149818,  0.08051826, ...,  0.16280383,
          0.0749408 , -0.0422778 ],
        [ 0.05935276, -0.18167965, -0.07092974, ..., -0.1289912 ,
          0.1795451 , -0.07565486],
        [-0.03963319,  0.06747865, -0.00316706, ..., -0.0013548 ,
         -0.19950242,  0.10193568],
        ...,
        [-0.10679527

In [5]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 5s 3us/step


In [6]:
# >>> Helper Function <<<

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

def preprocess_review(review):
    words = review.lower().split()
    encoded_review = [word_index.get(word, -1) + 3 for word in words]
    encoded_review = [1] + encoded_review
    padded_review = pad_sequences([encoded_review], maxlen=300, padding='pre')
    return padded_review

In [7]:
# >>> Prediction Function <<<

def predict_sentiment(review):
    preprocessed_review = preprocess_review(review)

    prediction = model.predict(preprocessed_review)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment, prediction[0][0] 

In [8]:
# >>> User Input and Prediction <<<

example_review = "An absolute masterpiece with a gripping narrative and stellar performances by the entire cast."

sentiment, score = predict_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Score: {score}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step
Review: An absolute masterpiece with a gripping narrative and stellar performances by the entire cast.
Sentiment: Positive
Score: 0.9946517944335938


In [9]:
# >>> Cleanup <<<

import gc
import torch 

gc.collect()

torch.cuda.empty_cache()